# WESAD · Selección exhaustiva de características | 10 sujetos TRAIN

**Alcance:** reducir las **261 características** con `q_FDR_BH < 0,05`, obtenidas en el análisis inferencial previo sobre los diez participantes S2–S11. **S13–S17 quedan totalmente reservados y nunca se leen.**

**No se utiliza muestreo ni submuestreo:** MI y correlaciones consideran **todas las ventanas** disponibles de los 10 sujetos y las 261 características. Para ahorrar RAM se emplean matrices `float32` mapeadas en disco, procesamiento secuencial, checkpoints, reutilización de resultados y un único proceso de cómputo.

### Flujo reproducible

1. Verificación de 261 características previamente significativas y catálogo de códigos de la tesis.
2. Ensamblaje incremental de una matriz de entrenamiento **solo con las 261 características**, sin cargar simultáneamente todos los sujetos.
3. Recuperación de etiquetas originales por ventana desde `metadata['original_label']` de los **checkpoints TRAIN**: WESAD 1=Baseline, 2=Stress, 3=Amusement. Se verifican contra el target binario existente. **No se inventan etiquetas**.
4. Estimación no paramétrica de información mutua (MI) para las **tres clases**, y MI binaria como análisis de sensibilidad. Ranking completo.
5. Matrices **Pearson y Spearman** sobre *todas* las ventanas de entrenamiento válidas en cada par de variables. Nueve mapas 30×30 (los 3 bloques contra los 3 bloques) con ambas cifras en cada casilla.
6. Aristas de redundancia si `|Pearson| ≥ 0.80` **o** `|Spearman| ≥ 0.80`. Selección voraz ordenada por mayor MI, conservando variables sin redundancias respecto de las ya conservadas.
7. Histograma de MI de las sobrevivientes, retención del percentil 75 superior (configurable). Listas completas de eliminación y conjunto final.
8. Auditoría de conteos por modalidad y por **familia de característica**; exportación de CSV/XLSX e imágenes, reanudación.

> **Advertencia metodológica:** las 261 características provienen de un cribado **binario** inicial; la MI triclase no convierte retrospectivamente esa selección en un test triclase. El solapamiento elevado de ventanas hace que MI y correlaciones sean descriptivos de entrenamiento y que su capacidad de generalización deba evaluarse con validación por sujeto. La elección de umbrales es una regla de reducción de complejidad, no una garantía de que se retiene el subconjunto óptimo.

**Lectura de `Perc_`:** porcentaje de características retenidas según el ranking de información mutua DESPUÉS del filtrado de redundancia. `Perc_=100` retiene todas las supervivientes; `Perc_=75` retiene aproximadamente el 75 % superior (redondeo hacia arriba). No es el percentil del valor de MI.


In [ ]:
from pathlib import Path
import os, gc, json, pickle, hashlib, math, warnings, tempfile, shutil, time
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_classif
from IPython.display import display, Image

TRAIN_SUBJECTS = [f'S{i}' for i in range(2,12)]
TEST_RESERVED = [f'S{i}' for i in range(13,18)]
BASE_DIR = Path.cwd()
ROOT = BASE_DIR / 'wesad_feature_output'
FULL_DIR = ROOT / 'checkpoints_completos'
PREV_OUT = ROOT / 'Analisis_univariado_bivariado_train_10_sujetos'
PREV_CACHE = PREV_OUT / 'cache_memmap_train'
SELECTION_DIR = ROOT / 'Seleccion_MI_Correlacion_10_Sujetos_261_r90_p100_m33x33'
MATRIX_DIR = SELECTION_DIR / 'cache_matrices_train'
FIG_DIR = SELECTION_DIR / 'figuras'
TABLE_DIR = SELECTION_DIR / 'tablas'
for path in (SELECTION_DIR, MATRIX_DIR, FIG_DIR, TABLE_DIR): path.mkdir(parents=True,exist_ok=True)

# Parámetros editables:
TARGET_MODE = 'multiclase'  # 'multiclase' (1,2,3) o 'binario' (0,1)
CORR_THRESHOLD = 0.90
### Las señales fisiológicas contienen variables relacionadas, pero no necesariamente equivalentes.
### Dos características pueden correlacionarse mucho y aun así representar propiedades fisiológicas diferentes.
### 0.9 es Moderada	Conserva más información fisiológica Mantiene algunas variables muy correlacionadas
### La razón es que ya redujiste las 455 características iniciales a 261 con diferencias estadísticamente significativas,
## y ahora buscamos conservar suficiente información fisiológica de ECG, EDA y RESP antes del entrenamiento. 
# Un umbral de 0,80 podría eliminar características que todavía aporten información complementaria por eso 0.9 es buena opcion


Perc_ = 100                   # porcentaje exacto a CONSERVAR del ranking MI: 100=todas, 75=top 75%, 50=top 50%
MI_NEIGHBORS = 3
SEED = 2026
MIN_VALID_FRACTION = 0.60    # excluir solo variables con < 60% finitos (sin muestrear observaciones)
SHOW_ALL_CORRELATION_FIGURES = True   # 45 matrices de hasta 30x30 (triángulo superior; sin duplicar pares)
SHOW_GLOBAL_FIGURE = True
FIG_DPI = 300
CHECKPOINT_MI_EVERY = 1
RECALCULATE_MI = False
RECALCULATE_CORRELATIONS = False
REGENERATE_FIGURES = False
MAX_PLOT_FONT = 11
CORR_BLOCK_SIZE = 30  # Matrices legibles para anexos; 261 variables -> 9 grupos y 45 matrices únicas

assert 0 <= Perc_ <= 100
assert 0 < CORR_THRESHOLD <= 1
assert TARGET_MODE in ('multiclase','binario')
np.random.seed(SEED)
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)
pd.set_option('display.width',240)
print('TRAIN:', TRAIN_SUBJECTS, '| TEST INTACTO:',TEST_RESERVED)
print('TODAS las ventanas: sin muestreo, sin subsampling.')
print('Destino:',SELECTION_DIR)
print(f'Retención por ranking de MI: Perc_={Perc_}% | bloque de correlación: {CORR_BLOCK_SIZE} variables')
# Escritura segura para Windows: si Excel/OneDrive/antivirus bloquea un destino,
# guarda una versión ALTERNATIVA en lugar de interrumpir el análisis.
from uuid import uuid4

def csv_seguro(df, destino, **kwargs):
    destino=Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    temporal=destino.with_name(destino.stem+f'.escritura_{uuid4().hex[:12]}'+destino.suffix)
    df.to_csv(temporal, **kwargs)
    try:
        os.replace(temporal,destino)
        return destino
    except (PermissionError,OSError) as exc:
        alternativo=destino.with_name(destino.stem+f'.guardado_{uuid4().hex[:12]}'+destino.suffix)
        os.rename(temporal,alternativo)
        print(f'AVISO: destino bloqueado: {destino.name}. Se guardó versión alternativa: {alternativo.name}. Detalle: {exc}',flush=True)
        return alternativo

def cargar_csv_segun_firma(path, firma, columnas):
    if not path.exists():return {}
    try:
        df=pd.read_csv(path,encoding='utf-8-sig')
        if not set(columnas).issubset(df.columns):return {}
        if not df.firma_datos.astype(str).eq(str(firma)).all():return {}
        return {r['feature_name']:r for r in df.to_dict('records') if r['feature_name'] in FEATURES}
    except Exception as exc:
        print(f'No se pudo reutilizar {path.name}: {exc}',flush=True)
        return {}


## 1. Reutilizar los 455 resultados y construir el catálogo de las 261 características

Se exige el CSV original de las pruebas pareadas ya calculadas, se revisa que haya 455 nombres únicos y se identifica exactamente el conjunto `q < 0.05`. El código de tesis (`ECG_49`, etc.) **no** se vuelve a inventar ni se confunde con el orden físico de las columnas. Si los archivos del estudio exploratorio están en otra ruta, edite únicamente `PREV_OUT`.

In [ ]:
STATS_FILE = PREV_OUT / 'resumen_estadistico_455.csv'
if not STATS_FILE.exists():
    raise FileNotFoundError(
        f'No se encontró {STATS_FILE}. Ajuste PREV_OUT a la carpeta de resultados del notebook anterior; '
        'no se recomputan p/q sin datos originales.')
results455 = pd.read_csv(STATS_FILE,encoding='utf-8-sig')
required={'codigo','feature_name','q_FDR_BH','p_pareada_exacta','significativa_FDR_0_05'}
assert required.issubset(results455.columns), required-set(results455.columns)
assert len(results455)==455 and results455['feature_name'].is_unique and results455['codigo'].is_unique
assert results455['codigo'].str.match(r'^(ECG|EDA|RESP)_\d+$').all()
selected261 = results455.loc[pd.to_numeric(results455.q_FDR_BH,errors='coerce') < .05].copy()
assert len(selected261)==261, f'Se esperaban 261 significativas, se encontraron {len(selected261)}. Revisar archivo de origen.'
selected261['senal']=selected261.codigo.str.split('_').str[0]
selected261['orden_codigo']=selected261.codigo.str.split('_').str[-1].astype(int)
selected261=selected261.sort_values(['senal','orden_codigo'],key=lambda c:c.map({'ECG':0,'EDA':1,'RESP':2}) if c.name=='senal' else c).reset_index(drop=True)
FEATURES=selected261.feature_name.tolist()
CODES=selected261.codigo.tolist()
assert len(FEATURES)==len(set(FEATURES))==261
selection_signature=hashlib.sha256(json.dumps({'features':FEATURES,'target':TARGET_MODE,'MI_NEIGHBORS':MI_NEIGHBORS,'SEED':SEED},sort_keys=True).encode()).hexdigest()[:16]
print('Significativas recuperadas:',len(FEATURES),'|',selected261.senal.value_counts().to_dict())
display(selected261[['codigo','feature_name','q_FDR_BH']].head(12))

## 2. Ensamblar la matriz completa mediante mmap

Primero se intenta reutilizar la caché previamente creada por el notebook exploratorio (`X_features_first.npy`, forma 455 × ventanas, por sujeto). Si falta, se construye a partir del checkpoint completo de **ese único sujeto**. La nueva matriz consolidada contiene 261 columnas en `float32`, sin ninguna ventana omitida. Cada sujeto aporta además su ID y ambas etiquetas.

**Etiquetas multiclase:** `metadata['original_label']` se recupera de los checkpoints completos de TRAIN; se rechaza cualquier etiqueta inconsistente o desalineada. La etiqueta `1` en WESAD significa Baseline, `2` Stress y `3` Amusement. `target_binario=1` es Stress, y `0` es la unión de Baseline y Amusement. En la estimación multiclase usamos `original_label` real.

**Memoria/disco:** una lectura de un solo checkpoint por sujeto para recuperar etiquetas y, si fuera necesario, sus características. El cache consolidado se reconstruye si la firma de columnas o número de filas cambió. Nunca se abren archivos de S13–S17.
**Compatibilidad Windows:** los archivos `.npy` mapeados se cierran explícitamente antes de renombrarlos. Si un bloqueo interrumpe el renombrado, el reintento valida las matrices parciales completas y recupera los archivos sin reconstruirlos. Si falla la validación, se reconstruyen de forma segura.


In [ ]:
def atomic_json(path,data):
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
    os.replace(tmp,path)

def get_checkpoint(subject):
    assert subject in TRAIN_SUBJECTS
    path=FULL_DIR/f'{subject}_features_full.pkl'
    if not path.exists():raise FileNotFoundError(path)
    with open(path,'rb') as fp: obj=pickle.load(fp)
    assert obj.get('subject')==subject
    return obj

def subject_paths(subject):
    assert subject in TRAIN_SUBJECTS
    d=PREV_CACHE/subject
    return d/'X_features_first.npy',d/'y.npy',d/'manifest.json'

def label_paths(subject):
    return MATRIX_DIR/f'{subject}_original_label.npy'

def get_labels(subject, force=False):
    label_path=label_paths(subject)
    xsrc,ysrc,manifest=subject_paths(subject)
    binary=np.load(ysrc,mmap_mode='r') if ysrc.exists() else None
    if label_path.exists() and not force:
        orig=np.load(label_path,mmap_mode='r')
        if binary is not None and len(orig)==len(binary):
            assert np.array_equal((np.asarray(orig)==2).astype(np.int8),np.asarray(binary))
            return np.asarray(orig,dtype=np.int8),np.asarray(binary,dtype=np.int8)
        del orig
    obj=get_checkpoint(subject)
    try:
        orig=np.asarray(obj['metadata']['original_label']).reshape(-1).astype(np.int8)
        y=np.asarray(obj['target']).reshape(-1).astype(np.int8)
        assert set(np.unique(orig))=={1,2,3},f'{subject}: no están las 3 clases originales {set(np.unique(orig))}'
        assert len(orig)==len(y) and np.array_equal((orig==2).astype(np.int8),y)
        if binary is not None:assert np.array_equal(y,binary)
        tmp=label_path.with_suffix('.tmp.npy')
        np.save(tmp,orig);os.replace(tmp,label_path)
        return orig,y
    finally:del obj;gc.collect()

def subject_feature_view(subject, expected_names):
    xp,yp,mp=subject_paths(subject)
    if xp.exists() and mp.exists():
        manifest=json.loads(mp.read_text(encoding='utf-8'))
        names=manifest.get('features',[])
        if len(names)==455 and set(FEATURES).issubset(set(names)):
            x=np.load(xp,mmap_mode='r')
            if x.ndim==2 and x.shape[0]==455 and x.shape[1]==len(expected_names):
                return x, [names.index(name) for name in FEATURES], None
            del x
    # Fallback: crea el pequeño bloque consolidado sin pasar por caché anterior.
    obj=get_checkpoint(subject)
    names=list(obj['feature_matrix']['names'])
    X=np.asarray(obj['feature_matrix']['X'])
    assert set(FEATURES).issubset(set(names))
    assert len(X)==len(expected_names)
    return X,[names.index(name) for name in FEATURES],obj

def close_memmaps(*arrays):
    """Cierra los handles de mmap antes de reemplazar archivos en Windows.

    Los objetos devueltos por np.load(..., mmap_mode=...) y open_memmap
    mantienen un handle de fichero mientras existe su mapeo.
    """
    for item in arrays:
        if item is None:
            continue
        item.flush() if isinstance(item, np.memmap) else None
        mm = getattr(item, '_mmap', None)
        if mm is not None:
            mm.close()

def replace_with_retry(src, dst, attempts=8):
    """Reintenta ante bloqueos transitorios de antivirus/indexación en Windows."""
    for attempt in range(attempts):
        try:
            os.replace(src, dst)
            return
        except PermissionError:
            if attempt == attempts - 1:
                raise PermissionError(
                    f'No fue posible reemplazar {src} -> {dst}. '
                    'Cierra otros kernels y programas que tengan abierto el .npy.'
                )
            gc.collect()
            time.sleep(min(0.25 * (attempt + 1), 2))

def recover_completed_partial_matrices(info, sizes, total):
    """Finaliza una ejecución interrumpida *solo* si TODOS los datos están escritos.

    Es seguro ejecutar esta verificación después del WinError 32 original;
    no modifica ni reconstruye los datos al comprobarlos.
    """
    pairs = [
        (MATRIX_DIR/'X_261_partial.npy', X_PATH),
        (MATRIX_DIR/'y_3_partial.npy', Y3_PATH),
        (MATRIX_DIR/'y_2_partial.npy', Y2_PATH),
        (MATRIX_DIR/'subject_partial.npy', SUBJECT_PATH),
    ]
    effective = [dst if dst.exists() else tmp for tmp, dst in pairs]
    if not all(p.exists() for p in effective):
        return False
    views = []
    try:
        for path in effective:
            views.append(np.load(path, mmap_mode='r'))
        x, y3, y2, subj = views
        if x.shape != (total, 261) or y3.shape != (total,) or y2.shape != (total,) or subj.shape != (total,):
            return False
        offset = 0
        for i, (entry, length) in enumerate(zip(info, sizes)):
            sl = slice(offset, offset + length)
            a = np.asarray(y3[sl])
            b = np.asarray(y2[sl])
            t = np.asarray(subj[sl])
            if not (np.all(np.isin(a, [1,2,3])) and
                    np.array_equal((a == 2).astype(np.int8), b) and
                    np.all(t == i) and
                    int((a == 1).sum()) == entry['baseline'] and
                    int((a == 2).sum()) == entry['stress'] and
                    int((a == 3).sum()) == entry['amusement']):
                return False
            offset += length
        if offset != total:
            return False
    except (OSError, ValueError, EOFError):
        return False
    finally:
        if views:
            close_memmaps(*views)
            del views
        gc.collect()
    print('✓ Se han verificado TODOS los registros de archivos parciales. Se finalizarán los nombres definitivos.')
    for tmp, dst in pairs:
        if not dst.exists():
            replace_with_retry(tmp, dst)
    atomic_json(MANIFEST_PATH, {
        'features': FEATURES, 'codes': CODES, 'subjects': TRAIN_SUBJECTS,
        'total_rows': total, 'per_subject': info, 'dtype': 'float32',
        'source': str(PREV_OUT), 'recovered_partial': True,
    })
    return True

MANIFEST_PATH=MATRIX_DIR/'manifest.json'
X_PATH=MATRIX_DIR/'X_261_complete_float32.npy'
Y3_PATH=MATRIX_DIR/'y_original_3clases.npy'
Y2_PATH=MATRIX_DIR/'y_binario.npy'
SUBJECT_PATH=MATRIX_DIR/'sujeto_idx.npy'

complete=False
if all(p.exists() for p in [MANIFEST_PATH,X_PATH,Y3_PATH,Y2_PATH,SUBJECT_PATH]):
    try:
        m=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        xchk=np.load(X_PATH,mmap_mode='r')
        ychk=np.load(Y3_PATH,mmap_mode='r')
        complete=(m['features']==FEATURES and m['subjects']==TRAIN_SUBJECTS and
                  xchk.shape==(m['total_rows'],261) and len(ychk)==m['total_rows'] and
                  all(p.exists() for p in [Y2_PATH,SUBJECT_PATH]))
        del xchk,ychk
    except Exception:complete=False

if not complete:
    # Tras un WinError 32 en os.replace, los temporales pueden estar completos.
    # Antes de volver a crear la matriz, intentamos recuperarlos.
    recovery_info=[]
    for subject in TRAIN_SUBJECTS:
        orig, bina = get_labels(subject)
        recovery_info.append({
            'sujeto': subject, 'n': len(orig),
            'baseline': int((orig == 1).sum()),
            'stress': int((orig == 2).sum()),
            'amusement': int((orig == 3).sum()),
        })
        del orig, bina
    recovery_sizes=[e['n'] for e in recovery_info]
    if recover_completed_partial_matrices(recovery_info, recovery_sizes, int(sum(recovery_sizes))):
        complete = True
    del recovery_info, recovery_sizes
    gc.collect()

if not complete:
    info=[]
    for subject in TRAIN_SUBJECTS:
        orig,bina=get_labels(subject)
        info.append({'sujeto':subject,'n':len(orig),'baseline':int((orig==1).sum()),'stress':int((orig==2).sum()),'amusement':int((orig==3).sum())})
        del orig,bina;gc.collect()
    sizes=[e['n'] for e in info]
    total=int(sum(sizes))
    print('Total ventanas de entrenamiento:',total)
    # Los temporales .npy hacen que una interrupción nunca sea confundida con caché completa.
    for p in [MANIFEST_PATH]:
        if p.exists():p.unlink()
    tmp_x=X_PATH.with_name('X_261_partial.npy')
    tmp_y3=Y3_PATH.with_name('y_3_partial.npy')
    tmp_y2=Y2_PATH.with_name('y_2_partial.npy')
    tmp_s=SUBJECT_PATH.with_name('subject_partial.npy')
    Xm=np.lib.format.open_memmap(tmp_x,'w+',dtype=np.float32,shape=(total,261))
    Y3=np.lib.format.open_memmap(tmp_y3,'w+',dtype=np.int8,shape=(total,))
    Y2=np.lib.format.open_memmap(tmp_y2,'w+',dtype=np.int8,shape=(total,))
    Ss=np.lib.format.open_memmap(tmp_s,'w+',dtype=np.uint8,shape=(total,))
    offset=0
    for si,(subject,length) in enumerate(zip(TRAIN_SUBJECTS,sizes)):
        orig,bina=get_labels(subject)
        data,inds,owner=subject_feature_view(subject,orig)
        for j in range(0,261,8):
            if data.shape[0]==455:
                Xm[offset:offset+length,j:j+8]=np.asarray(data[inds[j:j+8],:],dtype=np.float32).T
            else:
                Xm[offset:offset+length,j:j+8]=np.asarray(data[:,inds[j:j+8]],dtype=np.float32)
        Y3[offset:offset+length]=orig
        Y2[offset:offset+length]=bina
        Ss[offset:offset+length]=si
        offset+=length
        print(subject,':',length,'ventanas copiados |',f'{offset}/{total}',flush=True)
        del data,owner,orig,bina;gc.collect()
    # IMPORTANTE en Windows: no usar «for a in [Xm,...]» sin borrar «a».
    # Esa variable conservaba una referencia a Ss y bloqueaba subject_partial.npy.
    close_memmaps(Xm, Y3, Y2, Ss)
    del Xm, Y3, Y2, Ss
    gc.collect()
    for partial, final in [(tmp_x,X_PATH),(tmp_y3,Y3_PATH),
                           (tmp_y2,Y2_PATH),(tmp_s,SUBJECT_PATH)]:
        replace_with_retry(partial, final)
    atomic_json(MANIFEST_PATH,{'features':FEATURES,'codes':CODES,'subjects':TRAIN_SUBJECTS,'total_rows':total,'per_subject':info,'dtype':'float32','source':str(PREV_OUT)})
    gc.collect()
else:
    print('✓ Matriz completa previamente ensamblada, se reutiliza sin recalcular.')
manifest=json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['features']==FEATURES and manifest['subjects']==TRAIN_SUBJECTS
X=np.load(X_PATH,mmap_mode='r')
y3=np.load(Y3_PATH,mmap_mode='r')
y2=np.load(Y2_PATH,mmap_mode='r')
subidx=np.load(SUBJECT_PATH,mmap_mode='r')
assert X.shape[1]==261 and len(X)==len(y3)==len(y2)==len(subidx)
assert np.array_equal((np.asarray(y3)==2).astype(np.int8),np.asarray(y2))
assert set(np.unique(y3))=={1,2,3} and set(np.unique(y2))=={0,1}
print('✓ Shape completo:',X.shape,'| clases originales:',dict(zip(*np.unique(y3,return_counts=True))))
display(pd.DataFrame(manifest['per_subject']))

## 3. Control de calidad: todas las ventanas, sin descartar filas globales

Se calcula para cada variable su porcentaje de valores finitos y el número de valores únicos. Para MI se imputa **solo los no finitos** con la mediana de la característica calculada **exclusivamente en train**, manteniendo todas las ventanas y su clase. Para correlación, se usan **pares completos** (ambas características finitas), evitando fabricar correlaciones por imputación. Las variables con insuficiente cobertura o sin variación real no se incorporan al proceso de MI; quedan en una lista explícita de depuración. Estas decisiones se aplican a los diez sujetos de TRAIN; en validaciones por sujeto deben volver a ajustarse dentro del pliegue correspondiente.

In [ ]:
QC_PATH=TABLE_DIR/'control_calidad_261.csv'
if QC_PATH.exists():
    qc=pd.read_csv(QC_PATH,encoding='utf-8-sig')
    if qc.feature_name.tolist()!=FEATURES:qc=None
else: qc=None
if qc is None:
    qrows=[]
    for j,(name,code_) in enumerate(zip(FEATURES,CODES)):
        arr=np.asarray(X[:,j]);valid=arr[np.isfinite(arr)]
        med=float(np.median(valid)) if len(valid) else np.nan
        unique=int(np.unique(valid).size) if len(valid) else 0
        qrows.append({'codigo':code_,'feature_name':name,'senal':code_.split('_')[0],
                      'n_ventanas':len(arr),'n_finitos':len(valid),'frac_finitos':len(valid)/len(arr),
                      'mediana_imputacion_train':med,'n_unicos_finitos':unique,
                      'elegible':bool(len(valid)/len(arr)>=MIN_VALID_FRACTION and unique>1)})
        if (j+1)%50==0:print('QC:',j+1,'/261',flush=True)
    qc=pd.DataFrame(qrows)
    csv_seguro(qc,QC_PATH,index=False,encoding='utf-8-sig')
qc['elegible']=(qc.frac_finitos>=MIN_VALID_FRACTION)&(qc.n_unicos_finitos>1)
assert qc.feature_name.tolist()==FEATURES
bad=qc.loc[~qc.elegible].copy()
print('Características elegibles:',qc.elegible.sum(),'| en depuración:',len(bad))
display(bad)

## 4. Información mutua: estimación exhaustiva y ranking

Para clases discretas y una característica continua, la MI es \(I(X;Y)=\sum_y\int p(x,y)\log\frac{p(x,y)}{p(x)p(y)}\,dx\). La estimamos con k-vecinos más próximos (`mutual_info_classif`), manteniendo **todas las filas TRAIN** y una sola variable por iteración para evitar agotar memoria. El número de vecinos y la semilla son configurables; la MI se expresa en **nats**, no en porcentaje.

Se calculan MI respecto de tres condiciones (Baseline, Stress, Amusement) y respecto de la etiqueta binaria anterior. El **ranking y la eliminación** emplean `TARGET_MODE`; así se puede contrastar una elección triclase frente a la tarea binaria original. Para variables con pocos valores únicos, se usa el tratamiento discreto de scikit-learn (en particular valores binarios); las demás se tratan como continuas.

⚠️ MI no debe interpretarse como causalidad ni validación independiente: las ventanas están altamente solapadas y hay heterogeneidad entre sujetos. Los resultados se limitan a TRAIN.
**Windows:** cada característica calculada se guarda en un JSON independiente dentro de `tablas/checkpoints_MI_por_feature`. Si algún CSV está abierto en Excel o sincronizado, no se sobrescribe forzosamente; se genera un nombre alternativo sin detener los cálculos. Los checkpoints previos, incluido el CSV de la ejecución interrumpida, se recuperan al reiniciar.


In [ ]:
MI_PATH=TABLE_DIR/'informacion_mutua_261_completa.csv'
MI_CHECKPOINT_DIR=TABLE_DIR/'checkpoints_MI_por_feature'
MI_CHECKPOINT_DIR.mkdir(parents=True,exist_ok=True)

def calculate_one_mi(column,target,median,unique_count):
    a=np.asarray(column,dtype=np.float64).copy()
    bad=~np.isfinite(a)
    a[bad]=median
    discrete=unique_count<=2
    if discrete:
        _,a=np.unique(a,return_inverse=True)
    v=mutual_info_classif(a.reshape(-1,1),np.asarray(target,dtype=np.int8),
                          discrete_features=[discrete],n_neighbors=MI_NEIGHBORS,
                          random_state=SEED,copy=True)
    return float(v[0])

def mi_checkpoint_path(j,name):
    # El índice y nombre real protegen frente a cambios accidentales de orden.
    return MI_CHECKPOINT_DIR/f'{j+1:03d}_{name}.json'

def guardar_mi_individual(j,name,row):
    path=mi_checkpoint_path(j,name)
    # Archivo ya presente y válido: no modificarlo; evitar bloqueo por sobrescritura.
    if path.exists():
        try:
            prev=json.loads(path.read_text(encoding='utf-8'))
            if prev.get('firma_datos')==selection_signature and prev.get('feature_name')==name:
                return
        except Exception:pass
    temporal=path.with_name(path.stem+f'.nuevo_{uuid4().hex[:12]}'+path.suffix)
    temporal.write_text(json.dumps(row,ensure_ascii=False,allow_nan=True),encoding='utf-8')
    try:
        os.replace(temporal,path)
    except (PermissionError,OSError):
        # Guardado alternativo: también se recupera por el patrón del prefijo.
        print('Checkpoint principal bloqueado, conservando archivo alternativo:',temporal.name,flush=True)

def recuperar_mi_individual(j,name):
    path=mi_checkpoint_path(j,name)
    archivos=[path]+sorted(MI_CHECKPOINT_DIR.glob(f'{j+1:03d}_{name}.nuevo_*.json'),reverse=True)
    for file in archivos:
        if not file.exists():continue
        try:
            r=json.loads(file.read_text(encoding='utf-8'))
            if r.get('firma_datos')==selection_signature and r.get('feature_name')==name and r.get('codigo')==CODES[j]:
                return r
        except Exception:continue
    return None

# Recuperación: CSV completo previo, CSV parcial, variantes de salida y checkpoints por variable.
mi_cache={}
if not RECALCULATE_MI:
    candidatos=[MI_PATH,MI_PATH.with_suffix('.partial.csv')]
    candidatos+=sorted(TABLE_DIR.glob('informacion_mutua_261_completa.guardado_*.csv'),reverse=True)
    for cp in candidatos:
        for name,r in cargar_csv_segun_firma(cp,selection_signature,['codigo','feature_name','MI_multiclase','MI_binario','firma_datos']).items():
            mi_cache.setdefault(name,r)
    for j,name in enumerate(FEATURES):
        r=recuperar_mi_individual(j,name)
        if r is not None:mi_cache[name]=r
print(f'MI recuperadas de ejecuciones anteriores: {len(mi_cache)}/261',flush=True)

records=[]
for j,name in enumerate(FEATURES):
    if name in mi_cache and not RECALCULATE_MI:
        row=mi_cache[name]
    else:
        meta=qc.iloc[j]
        row={'codigo':CODES[j],'feature_name':name,'senal':meta['senal'],
             'firma_datos':selection_signature,'elegible':bool(meta['elegible']),
             'MI_multiclase':np.nan,'MI_binario':np.nan}
        if bool(meta['elegible']):
            row['MI_multiclase']=calculate_one_mi(X[:,j],y3,float(meta['mediana_imputacion_train']),int(meta['n_unicos_finitos']))
            row['MI_binario']=calculate_one_mi(X[:,j],y2,float(meta['mediana_imputacion_train']),int(meta['n_unicos_finitos']))
        mi_cache[name]=row
        guardar_mi_individual(j,name,row)   # persiste cada variable SIN sobrescribir CSV abierto
    records.append(row)
    if (j+1)%20==0 or j+1==len(FEATURES):
        print(f'MI: {j+1}/{len(FEATURES)} (checkpoints individuales)',flush=True)

mi_all=pd.DataFrame(records).merge(qc[['feature_name','frac_finitos','n_unicos_finitos']],on='feature_name',how='left')
mi_all['MI_seleccion']=mi_all['MI_multiclase'] if TARGET_MODE=='multiclase' else mi_all['MI_binario']
mi_all['elegible']=mi_all['feature_name'].map(qc.set_index('feature_name')['elegible'])
mi_rank=mi_all.sort_values(['MI_seleccion','codigo'],ascending=[False,True],na_position='last').reset_index(drop=True)
mi_rank.insert(0,'ranking_MI',np.arange(1,len(mi_rank)+1))
MI_SAVED_PATH=csv_seguro(mi_all,MI_PATH,index=False,encoding='utf-8-sig')
csv_seguro(mi_rank,TABLE_DIR/'ranking_MI_261.csv',index=False,encoding='utf-8-sig')
print('MI usada para selección:',TARGET_MODE)
print('MI consolidada guardada en:',MI_SAVED_PATH)
display(mi_rank)


## 5. Correlaciones completas de Pearson y Spearman sobre las 261 características

Se calculan las matrices **261 × 261** con todas las ventanas de los diez sujetos de entrenamiento. Las letras `p` y `s` en las celdas representan **Pearson y Spearman** (no valores *p* estadísticos). Se considera un par redundante si `|Pearson| ≥ CORR_THRESHOLD` **o** `|Spearman| ≥ CORR_THRESHOLD`. La comparación incluye correlaciones positivas y negativas.

Para representar todos los pares sin letras diminutas se particionan las 261 variables en **9 grupos** consecutivos de hasta **30** características (ocho grupos de 30 y el último de 21). Una matriz simétrica tiene información repetida debajo de la diagonal; por ello se exporta y muestra únicamente la **triangular superior de bloques: 9 diagonales + 36 cruces = 45 figuras**. Las 45 figuras incluyen **todas las relaciones únicas**, incluso entre grupos diferentes. Las matrices 261 × 261 también se guardan como CSV.


In [ ]:
PEARSON_PATH=MATRIX_DIR/'pearson_261.npy'
SPEARMAN_PATH=MATRIX_DIR/'spearman_261.npy'
if not (PEARSON_PATH.exists() and SPEARMAN_PATH.exists()) or RECALCULATE_CORRELATIONS:
    # DataFrame float32 mantiene el tamaño contenido; corr emplea internamente precisión doble.
    df=pd.DataFrame(X,columns=FEATURES,copy=False)
    print('Pearson completo, pairwise complete...',flush=True)
    pearson=df.corr(method='pearson',min_periods=3).to_numpy(dtype=np.float32)
    print('Spearman completo, pairwise complete...',flush=True)
    spearman=df.corr(method='spearman',min_periods=3).to_numpy(dtype=np.float32)
    np.save(PEARSON_PATH.with_name('pearson_partial.npy'),pearson)
    np.save(SPEARMAN_PATH.with_name('spearman_partial.npy'),spearman)
    try:
        os.replace(PEARSON_PATH.with_name('pearson_partial.npy'),PEARSON_PATH)
        os.replace(SPEARMAN_PATH.with_name('spearman_partial.npy'),SPEARMAN_PATH)
    except PermissionError as exc:
        raise PermissionError('Archivo NPY bloqueado: cierre visores de matrices y reinicie el kernel antes de recalcular correlaciones.') from exc
    del df,pearson,spearman;gc.collect()
pearson=np.load(PEARSON_PATH,mmap_mode='r')
spearman=np.load(SPEARMAN_PATH,mmap_mode='r')
assert pearson.shape==spearman.shape==(261,261)
assert np.allclose(pearson,pearson.T,equal_nan=True,atol=1e-5)
assert np.allclose(spearman,spearman.T,equal_nan=True,atol=1e-5)
pd.DataFrame(pearson,index=CODES,columns=CODES).to_csv(TABLE_DIR/'matriz_pearson_261.csv',encoding='utf-8-sig')
pd.DataFrame(spearman,index=CODES,columns=CODES).to_csv(TABLE_DIR/'matriz_spearman_261.csv',encoding='utf-8-sig')
P=np.array(pearson);S=np.array(spearman)
mask=np.isfinite(P)&(np.abs(P)>=CORR_THRESHOLD) | (np.isfinite(S)&(np.abs(S)>=CORR_THRESHOLD))
np.fill_diagonal(mask,False)
iupper,jupper=np.where(np.triu(mask,k=1))
edge_rows=[{'feature_A':FEATURES[i],'codigo_A':CODES[i],'feature_B':FEATURES[j],'codigo_B':CODES[j],
            'pearson':float(P[i,j]),'spearman':float(S[i,j]),
            'abs_max':float(np.nanmax([abs(P[i,j]),abs(S[i,j])]))}
           for i,j in zip(iupper,jupper)]
edges=pd.DataFrame(edge_rows,columns=['feature_A','codigo_A','feature_B','codigo_B','pearson','spearman','abs_max'])
csv_seguro(edges,TABLE_DIR/'parejas_correlacion_alta.csv',index=False,encoding='utf-8-sig')
print('Parejas de características con correlación alta:',len(edges))
print('Variables involucradas:',int(np.count_nonzero(mask.any(axis=1))))
display(edges.sort_values('abs_max',ascending=False))

## 6. Atlas de correlaciones: 45 matrices de hasta 30 × 30

Cada celda incluye en fucsia (**Pearson**) y negro (**Spearman**). **Solo Spearman** determina el color, con escala fija de azul (−1) a blanco (0) y rojo (+1). Un **borde dorado grueso** señala `|p| ≥ CORR_THRESHOLD` **o** `|s| ≥ CORR_THRESHOLD`, excluyendo las autocorrelaciones de la diagonal principal. Las figuras se guardan en PNG y PDF vectorial, en la carpeta `figuras/correlaciones_bloques_30`. Se visualizan las 45 matrices en el notebook cuando `SHOW_ALL_CORRELATION_FIGURES=True`.

**Lectura para tesis:** cada figura identifica el intervalo de características en filas y columnas. Los bloques cruzados completan el análisis: mostrar solo los nueve bloques diagonales ocultaría las correlaciones entre grupos. Los pares muy correlacionados están, además, en `tablas/parejas_correlacion_alta.csv`.


In [ ]:
def feature_label(idx):
    return f"{CODES[idx]}: {FEATURES[idx]}"
def correlation_plot(row_ids, col_ids, out_stem, title, show=True):
    r = np.array(row_ids, dtype=int)
    c = np.array(col_ids, dtype=int)

    p = P[np.ix_(r, c)]
    s = S[np.ix_(r, c)]
    hit = mask[np.ix_(r, c)]

    nrow, ncol = len(r), len(c)

    # ==========================================================
    # 1) CASO DE MATRICES MUY GRANDES (por si alguna vez lo usas)
    # ==========================================================
    if max(nrow, ncol) > 115:
        from PIL import Image as PILImage, ImageDraw, ImageFont
        from matplotlib import colormaps

        cellpx = 62
        padx = 250
        pady = 260
        right = 170
        bottom = 130

        width = padx + ncol * cellpx + right
        height = pady + nrow * cellpx + bottom

        im = PILImage.new('RGB', (width, height), 'white')
        d = ImageDraw.Draw(im)

        fontpath = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
        fontbold = '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'

        try:
            fcell_p = ImageFont.truetype(fontbold, 14)
            fcell_s = ImageFont.truetype(fontbold, 14)
            flabel = ImageFont.truetype(fontpath, 15)
            ftitle = ImageFont.truetype(fontbold, 28)
            fnote = ImageFont.truetype(fontpath, 17)
        except OSError:
            fcell_p = fcell_s = flabel = ftitle = fnote = ImageFont.load_default()

        cmap = colormaps.get_cmap('coolwarm')

        pearson_color = '#A0006D'   # fucsia
        spearman_color = '#111827'  # negro casi

        for ix in range(nrow):
            for jx in range(ncol):
                a = float(p[ix, jx])
                b = float(s[ix, jx])

                rgba = cmap(np.clip((b + 1) / 2, 0, 1)) if np.isfinite(b) else (.85, .85, .85, 1)
                fill = tuple(int(t * 255) for t in rgba[:3])

                x0 = padx + jx * cellpx
                y0 = pady + ix * cellpx

                # celda
                d.rectangle((x0, y0, x0 + cellpx, y0 + cellpx),
                            fill=fill, outline='#a8a8a8', width=1)

                # cajita blanca detrás del texto
                d.rounded_rectangle((x0 + 5, y0 + 8, x0 + cellpx - 5, y0 + cellpx - 8),
                                    radius=6, fill=(255, 255, 255), outline=None)

                t1 = f'{a:+.2f}' if np.isfinite(a) else 'NA'
                t2 = f'{b:+.2f}' if np.isfinite(b) else 'NA'

                d.text((x0 + 10, y0 + 11), t1, font=fcell_p, fill=pearson_color)
                d.text((x0 + 10, y0 + 31), t2, font=fcell_s, fill=spearman_color)

                if hit[ix, jx] and (r[ix] != c[jx]):  # no resaltar diagonal
                    d.rectangle((x0 + 2, y0 + 2, x0 + cellpx - 2, y0 + cellpx - 2),
                                outline='#ff8c00', width=5)

        d.text((28, 18), title, font=ftitle, fill='#111827')
        d.text((28, 65),
               'Primera línea (fucsia): Pearson | Segunda línea (negro): Spearman',
               font=fnote, fill='#111827')
        d.text((28, 92),
               'Color de celda: solo Spearman (azul −1, blanco 0, rojo +1)',
               font=fnote, fill='#111827')
        d.text((28, 119),
               f'Marco naranja: |Pearson| ≥ {CORR_THRESHOLD:.2f} o |Spearman| ≥ {CORR_THRESHOLD:.2f} | Diagonal no resaltada',
               font=fnote, fill='#111827')

        # barra visual de color
        for t in range(160):
            rr = cmap(t / 159)
            rgb = tuple(int(z * 255) for z in rr[:3])
            d.rectangle((padx + t * 3, 165, padx + t * 3 + 3, 179), fill=rgb)
        d.text((padx, 185),
               'Escala Spearman: −1                              0                              +1',
               font=fnote, fill='#111827')

        for ix in range(nrow):
            y0 = pady + ix * cellpx + 18
            d.text((10, y0), CODES[r[ix]], font=flabel, fill='#111827')

        for jx in range(ncol):
            x0 = padx + jx * cellpx + cellpx // 2
            label_text = CODES[c[jx]]
            text_layer = PILImage.new('RGBA', (180, 34), (255, 255, 255, 0))
            ImageDraw.Draw(text_layer).text((2, 2), label_text, font=flabel, fill='#111827')
            rotated = text_layer.rotate(90, expand=True)
            im.paste(rotated, (x0 - 16, pady - rotated.height - 8), rotated)

        png = FIG_DIR / f'{out_stem}.png'
        pdf = FIG_DIR / f'{out_stem}.pdf'
        if REGENERATE_FIGURES or not png.exists():
            im.save(png, optimize=False)
        if REGENERATE_FIGURES or not pdf.exists():
            im.save(pdf, 'PDF', resolution=180.0)

        del im, d
        gc.collect()

        if show:
            print(f'{title} | PNG {png.name} | PDF zoomable {pdf.name}')
            display(Image(filename=str(png), width=1400))
        return

    # ==========================================================
    # 2) CASO NORMAL: BLOQUES 30x30
    # ==========================================================
    fw = max(28, 1.15 * ncol + 9)
    fh = max(24, 1.00 * nrow + 8)

    fig, ax = plt.subplots(figsize=(fw, fh))
    im = ax.imshow(s, cmap='coolwarm', vmin=-1, vmax=1,
                   interpolation='none', aspect='equal')
    
    # etiquetas
    ax.set_xticks(np.arange(ncol))
    ax.set_yticks(np.arange(nrow))
    
    ax.set_xticklabels(
        [feature_label(i) for i in c],
        rotation=90,
        fontsize=10,
        fontweight='bold'
    )
    
    ax.set_yticklabels(
        [feature_label(i) for i in r],
        fontsize=10,
        fontweight='bold'
    )

    # rejilla
    ax.set_xticks(np.arange(-.5, ncol, 1), minor=True)
    ax.set_yticks(np.arange(-.5, nrow, 1), minor=True)
    ax.grid(which='minor', color='#8b8b8b', linewidth=0.35, alpha=0.55)
    ax.tick_params(which='minor', bottom=False, left=False)

    pearson_color = '#A0006D'
    spearman_color = '#111827'

    # textos por celda
    for i in range(nrow):
        for j in range(ncol):
            pi = p[i, j]
            si = s[i, j]

            pearson_txt = f'{pi:+.2f}' if np.isfinite(pi) else 'NA'
            spearman_txt = f'{si:+.2f}' if np.isfinite(si) else 'NA'

            # fondo blanco translúcido para legibilidad
            bbox_text = dict(boxstyle='round,pad=0.15',
                             facecolor='white', edgecolor='none', alpha=0.85)

            # línea superior: Pearson
            ax.text(j, i - 0.16, pearson_txt,
                    ha='center', va='center',
                    fontsize=10.8, fontweight='bold',
                    color=pearson_color,
                    bbox=bbox_text, zorder=6)

            # línea inferior: Spearman
            ax.text(j, i + 0.16, spearman_txt,
                    ha='center', va='center',
                    fontsize=10.8, fontweight='bold',
                    color=spearman_color,
                    bbox=bbox_text, zorder=6)

            # marco naranja si supera umbral (excepto diagonal)
            if hit[i, j] and (r[i] != c[j]):
                ax.add_patch(
                    patches.Rectangle((j - .5, i - .5), 1, 1,
                                      fill=False, edgecolor='#FFD700',
                                      linewidth=3.0, zorder=7)
                )

    ax.set_title(title, fontsize=20, fontweight='bold', pad=30)

    cb = fig.colorbar(im, ax=ax, shrink=.78, pad=.015)
    cb.set_label('Color de celda: Spearman (azul −1 a rojo +1)', fontsize=13)
    cb.ax.tick_params(labelsize=11)

    fig.text(0.5, 0.965,
             'Primera línea en fucsia: Pearson | Segunda línea en negro: Spearman',
             ha='center', fontsize=14, fontweight='bold')

    fig.text(0.5, 0.942,
             f'Marco dorado: |Pearson| ≥ {CORR_THRESHOLD:.2f} o |Spearman| ≥ {CORR_THRESHOLD:.2f} | Diagonal no resaltada',
             ha='center', fontsize=13)

    #fig.subplots_adjust(left=.17, right=.92, bottom=.19, top=.90)
    fig.subplots_adjust(
        left=0.33,
        right=0.93,
        bottom=0.35,
        top=0.90
    )
    png = BLOCK_FIG_DIR / f'{out_stem}.png'
    pdf = BLOCK_FIG_DIR / f'{out_stem}.pdf'

    if REGENERATE_FIGURES or not png.exists():
        fig.savefig(png, dpi=FIG_DPI, bbox_inches='tight', pad_inches=.35)
    if REGENERATE_FIGURES or not pdf.exists():
        fig.savefig(pdf, bbox_inches='tight', pad_inches=.35)

    plt.close(fig)
    gc.collect()

    if show:
        print(f'{title} | PNG {png.name} | PDF vectorial {pdf.name}')
        display(Image(filename=str(png), width=1500))

# Nueve grupos de máximo 30 características. Todos los pares únicos están en la triangular superior.
BLOCK_FIG_DIR=FIG_DIR / f'correlaciones_bloques_{CORR_BLOCK_SIZE}'
BLOCK_FIG_DIR.mkdir(parents=True,exist_ok=True)
blocks=[np.arange(i,min(i+CORR_BLOCK_SIZE,len(FEATURES)),dtype=int)
        for i in range(0,len(FEATURES),CORR_BLOCK_SIZE)]
num_figures=len(blocks)*(len(blocks)+1)//2
print(f'Grupos: {len(blocks)}; matrices únicas (sin duplicar espejo): {num_figures}.')
manifest_blocks=[]
for a,rr in enumerate(blocks):
    for b in range(a,len(blocks)):
        cc=blocks[b]
        stem=f'correlacion_filas_{a+1:02d}_columnas_{b+1:02d}_{len(rr)}x{len(cc)}'
        title=(f'WESAD TRAIN · grupo filas {a+1}/{len(blocks)} '
               f'({CODES[rr[0]]}–{CODES[rr[-1]]}) frente a columnas {b+1}/{len(blocks)} '
               f'({CODES[cc[0]]}–{CODES[cc[-1]]})')
        correlation_plot(rr,cc,stem,title,show=SHOW_ALL_CORRELATION_FIGURES)
        manifest_blocks.append({'bloque_filas':a+1,'bloque_columnas':b+1,
          'variables_filas':len(rr),'variables_columnas':len(cc),
          'codigo_inicio_filas':CODES[rr[0]],'codigo_fin_filas':CODES[rr[-1]],
          'codigo_inicio_columnas':CODES[cc[0]],'codigo_fin_columnas':CODES[cc[-1]],
          'imagen':str(BLOCK_FIG_DIR/f'{stem}.png'),'pdf':str(BLOCK_FIG_DIR/f'{stem}.pdf')})
        print(f'Bloques {a+1}/{len(blocks)} × {b+1}/{len(blocks)} listos.',flush=True)
csv_seguro(pd.DataFrame(manifest_blocks),TABLE_DIR/'indice_atlas_correlaciones_30.csv',index=False,encoding='utf-8-sig')
assert len(manifest_blocks)==num_figures
print(f'✓ {num_figures} matrices de hasta {CORR_BLOCK_SIZE}×{CORR_BLOCK_SIZE} exportadas.')


## 7. Una matriz adicional exclusivamente con las variables que presentan correlación alta

Solo se conservan filas/columnas de variables que tienen **al menos una pareja distinta de sí mismas** por encima del umbral en Pearson o Spearman. La diagonal se excluye del criterio. Este conjunto puede ser grande; se exporta además un listado de **todas** las parejas que sobrepasan el umbral. La imagen completa usa color Spearman y texto p/s, exactamente como las nueve anteriores.

In [ ]:
involved=np.flatnonzero(mask.any(axis=1))
if len(involved):
    correlation_plot(involved,involved,'matriz_solo_features_correlacion_alta',
                    f'Variables con al menos una correlación alta · {len(involved)} características · |p| o |s| ≥ {CORR_THRESHOLD:.2f}',
                    show=SHOW_GLOBAL_FIGURE)
else:
    print('Ninguna pareja supera el umbral: no se genera la matriz filtrada.')

## 8. Eliminar redundancias por orden descendente de MI

Algoritmo voraz reproducible: ordenar las características elegibles de mayor a menor `MI_seleccion` (desempate por código); tomar la primera y eliminar todas las que estén altamente correlacionadas **con alguna ya conservada**. Esto produce un **conjunto independiente** en el grafo de redundancia: ningún par seleccionado supera el umbral. La característica eliminada se registra con **el código, MI y correlaciones del representante que la bloqueó**.

La selección voraz no es un óptimo combinatorio global y depende de que MI sea comparable entre características; su ventaja es respetar la prioridad MI solicitada y producir una explicación exacta de cada eliminación.

In [ ]:
mi_rank_ok=mi_rank.loc[mi_rank.elegible & mi_rank.MI_seleccion.notna()].copy()
order=[FEATURES.index(f) for f in mi_rank_ok.feature_name]
kept_indices=[];removed=[]
for k in order:
    blockers=[b for b in kept_indices if mask[k,b]]
    if blockers:
        # El primer bloqueador tiene el MI más alto, dado el orden de selección.
        b=blockers[0]
        removed.append({'codigo':CODES[k],'feature_name':FEATURES[k],
          'MI_eliminada':float(mi_all.set_index('feature_name').loc[FEATURES[k],'MI_seleccion']),
          'motivo':'Correlación alta con característica conservada de mayor MI',
          'conservada_codigo':CODES[b],'conservada_feature':FEATURES[b],
          'MI_conservada':float(mi_all.set_index('feature_name').loc[FEATURES[b],'MI_seleccion']),
          'pearson':float(P[k,b]),'spearman':float(S[k,b]),
          'criterio':f'|Pearson| ≥ {CORR_THRESHOLD:.2f} o |Spearman| ≥ {CORR_THRESHOLD:.2f}'})
    else:kept_indices.append(k)
removed_corr=pd.DataFrame(removed,columns=['codigo','feature_name','MI_eliminada','motivo','conservada_codigo','conservada_feature','MI_conservada','pearson','spearman','criterio'])
retained_after_corr=mi_rank_ok.set_index('feature_name').loc[[FEATURES[k] for k in kept_indices]].reset_index()
removed_qc=qc.loc[~qc.elegible].copy()
assert not mask[np.ix_(kept_indices,kept_indices)].any()
assert len(kept_indices)+len(removed_corr)+len(removed_qc)==261
csv_seguro(removed_corr,TABLE_DIR/'eliminadas_por_correlacion.csv',index=False,encoding='utf-8-sig')
csv_seguro(retained_after_corr,TABLE_DIR/'supervivientes_tras_correlacion.csv',index=False,encoding='utf-8-sig')
csv_seguro(removed_qc,TABLE_DIR/'no_elegibles_por_calidad.csv',index=False,encoding='utf-8-sig')
print('Después de correlación:',len(kept_indices),'| eliminadas por redundancia:',len(removed_corr),'| no elegibles:',len(removed_qc))
display(removed_corr.head(40))

## 9. Distribución de MI y retención del **top `Perc_` %**

Las variables no redundantes se ordenan de **mayor a menor información mutua**. `Perc_` expresa **el porcentaje de características que se conserva**, no un percentil del valor de MI:

- `Perc_ = 100`: se conservan **todas**, eliminaciones por MI = **0**.
- `Perc_ = 75`: se conserva el **75 % de mayor MI** y se elimina el **25 % inferior**.
- `Perc_ = 50`: se conserva el **50 % de mayor MI**.

El número a conservar se calcula como `ceil(N × Perc_ / 100)` (redondeo hacia arriba). Los empates en MI se resuelven por `codigo` en orden alfabético para producir una lista determinista. Se muestra el histograma con el umbral real del ranking seleccionado. **Cambiar `Perc_` no exige recalcular MI ni correlaciones.**


In [ ]:
candidate=retained_after_corr.copy().sort_values(
    ['MI_seleccion','codigo'],ascending=[False,True],kind='mergesort'
).reset_index(drop=True)
assert len(candidate)>0,'No quedaron variables elegibles.'
assert 0 <= Perc_ <= 100
n_candidate=len(candidate)
n_keep=int(np.ceil(n_candidate*Perc_/100.0))
final_features=candidate.iloc[:n_keep].copy().reset_index(drop=True)
removed_low_mi=candidate.iloc[n_keep:].copy().reset_index(drop=True)
final_features.insert(0,'ranking_final',np.arange(1,len(final_features)+1))
removed_low_mi['motivo']=f'Fuera del top {Perc_}% de características con mayor información mutua'
removed_low_mi.insert(0,'ranking_MI_tras_correlacion',np.arange(n_keep+1,n_candidate+1))
# Umbral en nats: frontera descriptiva, NO se aplica un segundo filtro por valor.
mi_cut=float(final_features['MI_seleccion'].iloc[-1]) if n_keep else float('nan')
mi_next=float(removed_low_mi['MI_seleccion'].iloc[0]) if len(removed_low_mi) else float('nan')
csv_seguro(final_features,TABLE_DIR/'features_finales_seleccionadas.csv',index=False,encoding='utf-8-sig')
csv_seguro(removed_low_mi,TABLE_DIR/'eliminadas_por_baja_MI.csv',index=False,encoding='utf-8-sig')
print(f'Perc_={Perc_}% de retención | Candidatas: {n_candidate} | Top conservado: {n_keep}')
print(f'Seleccionadas: {len(final_features)} | Eliminadas por MI: {len(removed_low_mi)}')
print(f'MI mínimo de las seleccionadas: {mi_cut:.7g} nats | MI máximo de las descartadas: {mi_next:.7g} nats')
assert len(final_features)+len(removed_low_mi)+len(removed_corr)+len(removed_qc)==261
assert len(final_features)==int(np.ceil(n_candidate*Perc_/100.0))
if Perc_==100:
    assert len(removed_low_mi)==0,'Perc_=100 debe conservar el 100% tras correlación.'
# Figura: distribución de MI + frontera del ranking de selección
fig,ax=plt.subplots(figsize=(13,6))
ax.hist(candidate.MI_seleccion,bins=min(35,max(10,int(np.sqrt(n_candidate)))),color='#628ca5',edgecolor='white')
if n_keep and n_keep<n_candidate:
    ax.axvline(mi_cut,color='#e57421',lw=2.8,
               label=f'Límite del top {Perc_}% = {mi_cut:.5g} nats')
else:
    ax.axvline(mi_cut if n_keep else candidate.MI_seleccion.min(),color='#e57421',lw=2.8,
               label=f'Perc_={Perc_}% | conservadas: {n_keep}/{n_candidate}')
ax.set_title('Información mutua de características no redundantes | TRAIN completo',fontsize=15,fontweight='bold')
ax.set_xlabel('Información mutua con '+('3 clases WESAD' if TARGET_MODE=='multiclase' else 'target binario')+' (nats)')
ax.set_ylabel('Número de características')
ax.grid(axis='y',alpha=.18);ax.legend();fig.tight_layout()
fig.savefig(FIG_DIR/'distribucion_MI_y_Perc_retencion.png',dpi=190)
plt.show();plt.close(fig)
print('Listado ordenado de características seleccionadas:')
display(final_features)
print('Listado ordenado de características eliminadas por MI:')
display(removed_low_mi)


## 10. Clasificación por familias fisiológicas/matemáticas y auditoría completa

Las familias se asignan **según el nombre técnico real** de la característica, sin modificar su valor. Se publican agregados por señal y cruzados por señal × familia, en las etapas inicial (261), posterior a correlación y final. También se genera una tabla de trazabilidad: **cada una de las 261 características tiene un único estado final**.

La clasificación de familias es analítica, no una taxonomía clínica universal. El criterio puede verificarse y refinarse sin cambiar la selección.

In [ ]:
def classify_family(name):
    s=name.lower()
    if s.startswith('ecg_'):
        if any(k in s for k in ['_hrv_']): return 'HRV espectral'
        if any(k in s for k in ['poincare','sd1','sd2']): return 'Poincaré / geométricas RR'
        if '_rr_' in s and any(k in s for k in ['entropy','complexity','_fd','dfa','rqa']):return 'Complejidad RR'
        if any(k in s for k in ['_hr_','_nn','_rmssd','_pnn','_sdsd','_rpeak','_rr_','_sdnn']):return 'Frecuencia cardíaca / HRV temporal'
        if 'wavelet' in s:return 'Wavelet ECG'
        if any(k in s for k in ['spectral','frequency','_power_','_relative_power_']):return 'Espectral ECG'
        return 'Estadística temporal ECG'
    if s.startswith('eda_'):
        if '_scr_' in s:return 'SCR / respuestas fásicas'
        if 'wavelet' in s:return 'Wavelet EDA'
        if any(k in s for k in ['entropy','complexity','_fd','dfa','rqa']):return 'Complejidad EDA'
        if 'spectral' in s or '_power_' in s or 'frequency' in s:return 'Espectral EDA'
        if '_tonic_' in s:return 'EDA tónica'
        if '_phasic_' in s:return 'EDA fásica'
        return 'Estadística temporal EDA cruda'
    if s.startswith('resp_'):
        if 'wavelet' in s:return 'Wavelet RESP'
        if any(k in s for k in ['entropy','complexity','_fd','dfa','rqa']):return 'Complejidad RESP'
        if any(k in s for k in ['frequency','spectral','_power_']):return 'Espectral RESP'
        if any(k in s for k in ['cycle','inspiration','expiration','breath','rate_per_minute','peak_count']):return 'Ciclos y frecuencia respiratoria'
        return 'Estadística temporal RESP'
    raise ValueError(name)

master=mi_rank[['codigo','feature_name','senal','MI_multiclase','MI_binario','MI_seleccion','elegible']].copy()
master['familia']=master.feature_name.map(classify_family)
statuses={f:'NO_ELEGIBLE_CALIDAD' for f in removed_qc.feature_name}
statuses.update({f:'ELIMINADA_CORRELACION' for f in removed_corr.feature_name})
statuses.update({f:'ELIMINADA_BAJA_MI' for f in removed_low_mi.feature_name})
statuses.update({f:'SELECCIONADA_FINAL' for f in final_features.feature_name})
master['estado_final']=master.feature_name.map(statuses)
assert master.estado_final.notna().all() and len(master)==261
master['tras_correlacion']=master.estado_final.isin(['ELIMINADA_BAJA_MI','SELECCIONADA_FINAL'])
master['final']=master.estado_final.eq('SELECCIONADA_FINAL')
csv_seguro(master,TABLE_DIR/'trazabilidad_completa_261.csv',index=False,encoding='utf-8-sig')

def aggregate(cols):
    g=master.groupby(cols,dropna=False).agg(
      inicial_261=('codigo','size'),
      no_elegibles=('estado_final',lambda x:int((x=='NO_ELEGIBLE_CALIDAD').sum())),
      eliminadas_correlacion=('estado_final',lambda x:int((x=='ELIMINADA_CORRELACION').sum())),
      tras_correlacion=('tras_correlacion','sum'),
      eliminadas_baja_MI=('estado_final',lambda x:int((x=='ELIMINADA_BAJA_MI').sum())),
      seleccionadas_finales=('final','sum'),
      MI_mediana=('MI_seleccion','median'),MI_maxima=('MI_seleccion','max')).reset_index()
    g['porcentaje_retenido']=100*g.seleccionadas_finales/g.inicial_261
    assert (g.inicial_261==(g.no_elegibles+g.eliminadas_correlacion+g.eliminadas_baja_MI+g.seleccionadas_finales)).all()
    return g

summary_signal=aggregate(['senal'])
summary_family=aggregate(['senal','familia'])
csv_seguro(summary_signal,TABLE_DIR/'resumen_por_senal.csv',index=False,encoding='utf-8-sig')
csv_seguro(summary_family,TABLE_DIR/'resumen_por_senal_y_familia.csv',index=False,encoding='utf-8-sig')
print('Resumen por señal:');display(summary_signal)
print('Resumen por señal y tipo:');display(summary_family)
print('Conteos generales:');display(master.estado_final.value_counts().rename_axis('estado').reset_index(name='n'))

## 11. Tablas Excel, manifiesto y validaciones finales

Se guardan todas las tablas como CSV (`UTF-8-SIG`, Excel en español) y en un libro XLSX de hojas identificables. Además se exportan: lista de pares altamente correlacionados, rankings MI, eliminaciones detalladas, final y auditoría por modalidad/familia. Se valida que cada característica de las 261 aparece exactamente una vez en el resultado final, y que las seleccionadas no tienen parejas `|p|` o `|s|` ≥ umbral.

**Importante para la tesis:** el filtro supervisado de 261 variables y la selección por MI deben repetirse **dentro de los pliegues internos** si se cuantifica rendimiento mediante validación cruzada por sujetos. El conjunto final aquí es exploratorio, fijo por los diez sujetos de TRAIN; los cinco TEST siguen completamente reservados para una sola evaluación final.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.dimensions import ColumnDimension
sheets={
 'Ranking_MI':mi_rank,
 'Pares_correlacion':edges,
 'Eliminadas_correlacion':removed_corr,
 'Tras_correlacion':retained_after_corr,
 'Eliminadas_baja_MI':removed_low_mi,
 'No_elegibles_calidad':removed_qc,
 'Seleccion_final':final_features,
 'Trazabilidad_261':master,
 'Resumen_por_senal':summary_signal,
 'Resumen_senal_familia':summary_family,
 'Control_calidad':qc,
}
book_path=TABLE_DIR/'seleccion_completa_261.xlsx'
wb=Workbook();wb.remove(wb.active)
for title,frame in sheets.items():
    ws=wb.create_sheet(title[:31]);ws.append(frame.columns.astype(str).tolist())
    for row in frame.itertuples(index=False,name=None):
        ws.append([None if pd.isna(v) else (v.item() if isinstance(v,np.generic) else v) for v in row])
    ws.freeze_panes='A2';ws.auto_filter.ref=ws.dimensions
    for cell in ws[1]:cell.fill=PatternFill('solid',fgColor='17324D');cell.font=Font(color='FFFFFF',bold=True)
    for j,cl in enumerate(frame.columns,start=1):
        width=min(58,max(12,len(str(cl))+2,int(frame[cl].astype(str).str.len().quantile(.80)+2) if len(frame) else 12))
        ws.column_dimensions[get_column_letter(j)].width=width
try:
    wb.save(book_path)
except PermissionError:
    book_path=book_path.with_name(book_path.stem+f'_guardado_{uuid4().hex[:8]}'+book_path.suffix)
    wb.save(book_path)
    print('Excel anterior abierto o bloqueado; guardado alternativo:',book_path)
report={
 'sujetos_train':TRAIN_SUBJECTS,'sujetos_test_no_accedidos':TEST_RESERVED,
 'total_ventanas_train':int(len(y3)),'inicial_significativas':261,
 'target_mi':TARGET_MODE,'clases_originales':{'1':'Baseline','2':'Stress','3':'Amusement'},
 'correlacion_umbral_abs_o':CORR_THRESHOLD,'porcentaje_retencion_mi_Perc_':Perc_,
 'mi_minima_seleccionada_nats':mi_cut,'mi_maxima_eliminada_nats':mi_next,'total_parejas_redundantes':int(len(edges)),'matrices_correlacion_bloques':len(manifest_blocks),'bloque_maximo_correlacion':CORR_BLOCK_SIZE,
 'no_elegibles':len(removed_qc),'eliminadas_correlacion':len(removed_corr),
 'eliminadas_baja_mi':len(removed_low_mi),'finales':len(final_features),
 'completo_sin_muestreo':True,'imputacion_MI':'mediana de train por feature',
 'correlacion':'pairwise complete, Pearson/Spearman, todas las ventanas',
 'nota':'exploratorio TRAIN; para CV por sujeto reentrenar selección dentro de cada fold'
}
atomic_json(SELECTION_DIR/'resumen_reproducibilidad.json',report)
print('✓ COMPLETADO SIN SUBMUESTREO')
for k,v in report.items():print(f'{k}: {v}')
print('Excel:',book_path)
print('Figuras:',FIG_DIR)
print('Tablas:',TABLE_DIR)

## 12. Features seleccionadas Correlacion y Mutual Information

In [ ]:
# ============================================================
# LISTADO FINAL DE CARACTERÍSTICAS SELECCIONADAS
# ============================================================

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

features_seleccionadas = (
    final_features
    .sort_values(
        ['MI_seleccion', 'codigo'],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

features_seleccionadas.insert(
    0, 'Orden',
    range(1, len(features_seleccionadas) + 1)
)

columnas = [
    'Orden',
    'codigo',
    'feature_name',
    'senal',
    'MI_seleccion'
]

print("=" * 75)
print("CARACTERÍSTICAS FINALES SELECCIONADAS")
print("=" * 75)

print(f"Total seleccionadas: {len(features_seleccionadas)}")
print(f"Umbral de correlación: {CORR_THRESHOLD}")
print(f"Porcentaje de retención por MI: {Perc_}%")

print("\nListado completo ordenado por información mutua:")

display(
    features_seleccionadas[columnas]
)

#print("\nNOMBRES DE LAS FEATURES PARA UTILIZAR EN LOS MODELOS:")

#FEATURES_FINALES = features_seleccionadas['feature_name'].tolist()

#for i, nombre in enumerate(FEATURES_FINALES, start=1):
#    print(f"{i:03d}. {nombre}")